# Certificate-Gated Defense Experiment
**Run the full attack/defense grid on Colab Pro A100 GPU**

This notebook:
1. Uploads the project code
2. Installs dependencies + loads **Qwen2.5-14B-Instruct** on A100 GPU
3. Runs 9 defenses × 5 seeds × 50 tasks = **2,250 episodes** with a real LLM
4. Computes metrics and generates result figures

Runtime: ~20-40 min on A100 40GB

## 0. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {name}")
    print(f"VRAM: {vram:.1f} GB")
    if vram < 30:
        print("⚠️  You have a smaller GPU. Go to Runtime > Change runtime type > A100 GPU")
        print("   Or change model_name below to 'Qwen/Qwen2.5-7B-Instruct' for T4/V100")
    else:
        print("✓ A100 detected — running Qwen2.5-14B-Instruct (best quality)")

## 1. Upload project
Upload your `cert-agent-exp` folder as a zip, or clone from GitHub.

In [ ]:
import os

# Option A: Upload zip (run this cell, then upload cert-agent-exp.zip)
# from google.colab import files
# uploaded = files.upload()  # upload cert-agent-exp.zip
# !unzip -qo cert-agent-exp.zip

# Option B: Clone from GitHub (uncomment and edit)
# !git clone https://github.com/YOUR_USER/cert-agent-exp.git

# Option C: Mount Google Drive (if you put the project there)
from google.colab import drive
drive.mount('/content/drive')
# Copy from Drive to local (faster I/O)
!cp -r /content/drive/MyDrive/cert-agent-exp /content/cert-agent-exp 2>/dev/null || echo "Put cert-agent-exp in your Google Drive root, or use Option A/B above"

os.chdir('/content/cert-agent-exp')
!ls

## 2. Install dependencies

In [ ]:
!pip install -q pydantic pyyaml tqdm rich orjson regex scikit-learn matplotlib \
    sentence-transformers faiss-cpu datasets torch transformers accelerate

In [ ]:
# Make the package importable
import sys
sys.path.insert(0, '/content/cert-agent-exp/src')

# Quick import test
from cert_agent_exp.models import generate
print("Imports OK")

## 3. Test the LLM on GPU

In [ ]:
import torch
vram = torch.cuda.get_device_properties(0).total_mem / 1e9 if torch.cuda.is_available() else 0

# Auto-select best model for your GPU
if vram >= 35:
    MODEL = "Qwen/Qwen2.5-14B-Instruct"   # A100 40GB — best quality
elif vram >= 20:
    MODEL = "Qwen/Qwen2.5-7B-Instruct"    # V100 / L4
else:
    MODEL = "Qwen/Qwen2.5-3B-Instruct"    # T4 fallback

print(f"Selected model: {MODEL}  (VRAM: {vram:.0f} GB)")
print("Loading model (first time downloads weights)...")

result = generate(
    'What is 2+2? Answer with just the number.',
    mode='hf',
    model_name=MODEL,
    temperature=0.0,
)
print(f"Model response: {result}")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 4. Configure the grid

In [ ]:
import yaml

config = {
    'data_dir': 'data',
    'runs_dir': 'runs',
    'retrieval_mode': 'faiss',
    'use_injected_corpus': True,
    'agent': {
        'type': 'retrieval_echo',
        'max_steps': 12,
    },
    'models': {
        'mode': 'hf',
        'model_name': MODEL,  # auto-selected in previous cell
        'temperature': 0.2,
        'seed': 0,
    },
    'grid': {
        'datasets': ['hotpotqa'],
        'defenses': [
            'none', 'quote_only', 'provenance_tags', 'allowlist',
            'quote+prov+allowlist', 'certificate_gating',
            'taskshield', 'llm_judge', 'intentguard',
        ],
        'channels': ['retrieval'],
        'strategies': ['non_adaptive'],
        'B_tokens': [50, 150, 300],
        'K_sources': [1, 2],
        'seeds': [0, 1, 2, 3, 4],
        'max_per_cell': 50,  # 9 defenses × 5 seeds × 50 = 2250 episodes
    },
}

os.makedirs('configs', exist_ok=True)
with open('configs/grid_colab.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

n_episodes = len(config['grid']['defenses']) * len(config['grid']['seeds']) * config['grid']['max_per_cell']
print(f"Grid: {n_episodes} total episodes")
print(f"Model: {MODEL} on GPU")
print(f"Estimated time: ~{n_episodes * 1.5 / 60:.0f}-{n_episodes * 3 / 60:.0f} min on A100")

## 5. Run the full grid

In [ ]:
import time
start = time.time()

!python scripts/05_run_grid.py --config configs/grid_colab.yaml

elapsed = time.time() - start
print(f"\n=== Grid completed in {elapsed/60:.1f} minutes ===")

## 6. Compute metrics

In [ ]:
!python scripts/06_compute_metrics.py --config configs/grid_colab.yaml

## 7. Generate figures

In [ ]:
!python scripts/07b_plot_performance.py --config configs/grid_colab.yaml

In [ ]:
from IPython.display import Image, display
import glob

for fig in sorted(glob.glob('runs/figures/*.png')):
    print(f"\n--- {os.path.basename(fig)} ---")
    display(Image(filename=fig, width=700))

## 8. Run internal proof package

In [ ]:
!python scripts/09_proof_package.py --config configs/grid_colab.yaml

In [ ]:
for fig in sorted(glob.glob('runs/proof/*.png')):
    print(f"\n--- {os.path.basename(fig)} ---")
    display(Image(filename=fig, width=700))

## 9. Live demo: watch attack & defense

In [ ]:
!python scripts/10_live_demo.py --config configs/grid_colab.yaml --defense none --fast

In [ ]:
!python scripts/10_live_demo.py --config configs/grid_colab.yaml --defense certificate_gating --fast

In [ ]:
!python scripts/10_live_demo.py --config configs/grid_colab.yaml --compare-all --fast

## 10. Sample LLM outputs (inspect what the model actually said)

In [ ]:
import json

logs = []
with open('runs/logs/grid_run.jsonl') as f:
    for line in f:
        logs.append(json.loads(line))

print(f"Total episodes: {len(logs)}\n")

# Show one example per defense
seen = set()
for L in logs:
    d = L.get('defense', '')
    if d in seen:
        continue
    seen.add(d)
    pa = L.get('parsed_action', {})
    content = (pa.get('content') or '')[:300]
    outcome = L.get('defense_trace', {}).get('final_outcome', '?')
    success = L.get('success', '?')
    print(f"=== {d} === outcome={outcome}  task_success={success}")
    print(f"  LLM output: {content}...")
    print()

## 11. Download results

In [ ]:
!zip -r results.zip runs/logs/ runs/metrics/ runs/figures/ runs/proof/ REPORT.md

from google.colab import files
files.download('results.zip')